In [ ]:
import torch
import tiktoken
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "LiquidAI/LFM2.5-350M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
model.eval()

assistant_name = "SkyFlight Assistant"
user_inquiry = "I would like to book a flight ticket from Jeddah to Doha"

token_ids = tokenizer.encode(user_inquiry)
tokens = [tokenizer.decode([i]) for i in token_ids]

print("✈️ Assistant:", assistant_name)
print("💬 User inquiry:", user_inquiry)
print("🧩 Tokens:", tokens)
print("🔢 Token IDs:", token_ids)
print("🔍 Number of tokens:", len(token_ids))

config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  709MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

✈️ Assistant: SkyFlight Assistant
💬 User inquiry: I would like to book a flight ticket from Jeddah to Doha
🧩 Tokens: ['<|startoftext|>', 'I', ' would', ' like', ' to', ' book', ' a', ' flight', ' ticket', ' from', ' J', 'edd', 'ah', ' to', ' D', 'oha']
🔢 Token IDs: [1, 550, 1499, 1672, 811, 2643, 768, 11398, 30437, 988, 962, 7459, 1285, 811, 889, 42854]
🔍 Number of tokens: 16


In [ ]:
system_prompt = """
You are a flight and aviation assistant.
Answer questions about flight schedules, destinations like Jeddah, Dammam, and Doha, and bookings.
Do not invent flight numbers or gate information.
If information is unavailable, say it is not confirmed.
"""

max_new_tokens = 80

def ask_model(question):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("🛫 Passenger:", user_inquiry)
print("✈️ Assistant:", ask_model(user_inquiry))

🛫 Passenger: I would like to book a flight ticket from Jeddah to Doha


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


✈️ Assistant: I'm sorry, but I don't have access to real-time flight schedules or booking systems. You'll need to use a travel website or contact an airline directly to book your flight from Jeddah to Doha.


In [ ]:
retries = 3
timeout_seconds = 120

def ask_with_retry(question):
    for attempt in range(1, retries + 1):
        try:
            inputs = tokenizer.apply_chat_template(
                [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
                add_generation_prompt=True, return_tensors="pt", return_dict=True
            )
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, max_time=timeout_seconds)
            print(f"✅ Attempt {attempt}: Success")
            return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        except Exception as e:
            print(f"⚠️ Attempt {attempt} failed:", e)
    return "Sorry, please try again later."

print("🤖 Assistant:", ask_with_retry("What are the flights to Dammam?"))

✅ Attempt 1: Success
🤖 Assistant: I don't have access to real-time flight schedules or booking information. You'll need to check a travel website or contact an airline directly for details about flights to Dammam.


In [ ]:
from pydantic import BaseModel, Field, ValidationError

class FlightBooking(BaseModel):
    passenger_name: str = Field(min_length=2)
    destination: str
    flight_class: str

booking_data = {"passenger_name": "Ahmed", "destination": "Jeddah", "flight_class": "Economy"}

try:
    booking = FlightBooking(**booking_data)
    print("✅ بيانات الحجز صحيحة")
    print("🛫 Booking Details:", booking.model_dump())
except ValidationError as e:
    print("❌ خطأ في بيانات الحجز:", e)

✅ بيانات الحجز صحيحة
🛫 Booking Details: {'passenger_name': 'Ahmed', 'destination': 'Jeddah', 'flight_class': 'Economy'}


In [ ]:
import asyncio
import time

def check_flight_status(name, flight_no):
    return f"Flight {flight_no} for {name}: On Time - Gate 4"

def list_destinations():
    return ["Jeddah", "Dammam", "Doha"]

async def run_tool(func, *args):
    await asyncio.sleep(0.5)
    return func(*args)

async def main():
    start = time.perf_counter()

    # تنفيذ العمليات غير المتزامنة مباشرة باستخدام await
    status, destinations = await asyncio.gather(
        run_tool(check_flight_status, "Ahmed", "SV-123"),
        asyncio.to_thread(list_destinations)
    )

    elapsed = time.perf_counter() - start

    print("🛫 Flight Status:", status)
    print("🌍 Destinations:", destinations)
    print(f"⏱️ Total Time: {elapsed:.2f} s")

# تشغيل الدالة الرئيسية بشكل يتوافق مع بيئة Colab
await main()

🛫 Flight Status: Flight SV-123 for Ahmed: On Time - Gate 4
🌍 Destinations: ['Jeddah', 'Dammam', 'Doha']
⏱️ Total Time: 0.50 s


In [ ]:
import re
from dataclasses import dataclass

aviation_text = """
Jeddah Route: Direct flights operate daily to King Abdulaziz International Airport.
Dammam Route: Regular scheduled flights available with regional connections.
Doha Route: Premium partner flights connecting seamlessly to international destinations.
"""

clean_text = re.sub(r"\s+", " ", aviation_text).strip()

@dataclass
class Document:
    source: str
    content: str

document = Document(source="airline_policy", content=clean_text)
print("📄 Source:", document.source)
print("🧹 Cleaned Text:", document.content)

📄 Source: airline_policy
🧹 Cleaned Text: Jeddah Route: Direct flights operate daily to King Abdulaziz International Airport. Dammam Route: Regular scheduled flights available with regional connections. Doha Route: Premium partner flights connecting seamlessly to international destinations.


In [ ]:
from sentence_transformers import SentenceTransformer

def recursive_chunking(text, chunk_size=30, overlap=5):
    words = text.split()
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]

recursive_chunks = recursive_chunking(document.content)
model_embed = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model_embed.encode(recursive_chunks)

print("🧩 Chunks Count:", len(recursive_chunks))
print("📐 Vector Dimensions:", embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🧩 Chunks Count: 2
📐 Vector Dimensions: 384


In [ ]:
# تثبيت المكتبة في حال لم تكن موجودة
!pip -q install chromadb

import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("flight_knowledge")

collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(recursive_chunks))],
    documents=recursive_chunks,
    embeddings=embeddings.tolist()
)

query = "Which destination has direct flights daily?"
query_vector = model_embed.encode([query]).tolist()
results = collection.query(query_embeddings=query_vector, n_results=1)

print("❓ Query:", query)
print("🔍 Retrieved Context:", results["documents"][0][0])

❓ Query: Which destination has direct flights daily?
🔍 Retrieved Context: Jeddah Route: Direct flights operate daily to King Abdulaziz International Airport. Dammam Route: Regular scheduled flights available with regional connections. Doha Route: Premium partner flights connecting seamlessly to international destinations.


In [ ]:
def flight_rag_service(question):
    q_vector = model_embed.encode([question]).tolist()
    res = collection.query(query_embeddings=q_vector, n_results=1)
    context = res["documents"][0][0]
    answer = f"According to flight guidelines: {context}"
    return context, answer

ctx, ans = flight_rag_service("Tell me about Jeddah flights")
print("📄 Context:", ctx)
print("🤖 Answer:", ans)

📄 Context: Jeddah Route: Direct flights operate daily to King Abdulaziz International Airport. Dammam Route: Regular scheduled flights available with regional connections. Doha Route: Premium partner flights connecting seamlessly to international destinations.
🤖 Answer: According to flight guidelines: Jeddah Route: Direct flights operate daily to King Abdulaziz International Airport. Dammam Route: Regular scheduled flights available with regional connections. Doha Route: Premium partner flights connecting seamlessly to international destinations.


In [ ]:
chat_messages = [
    "What is the flight status for SV-123?",
    "Can I select a window seat?",
    "Is online check-in open?",
    "Where is the boarding gate?",
    "How many bags can I check in?"
]

window_size = 3
recent_messages = chat_messages[-window_size:]

print("🪟 Recent Active Messages in Memory:")
for msg in recent_messages:
    print("✅", msg)

🪟 Recent Active Messages in Memory:
✅ Is online check-in open?
✅ Where is the boarding gate?
✅ How many bags can I check in?


In [ ]:
def observe(func):
    def wrapper(question):
        print("🔍 Aviation Trace Started...")
        result = func(question)
        print("✅ Trace Completed Successfully")
        return result
    return wrapper

@observe
def monitor_flight_assistant(question):
    return "Confirmed: Flight to Jeddah is on schedule."

print("🤖 Result:", monitor_flight_assistant("Check Jeddah flight"))

🔍 Aviation Trace Started...
✅ Trace Completed Successfully
🤖 Result: Confirmed: Flight to Jeddah is on schedule.
